In [ ]:
from pathlib import Path

import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

from topostats.damage.io import construct_grains_collection_from_topostats_files

# Config

In [ ]:
dir_base = Path("/Volumes/shared/pyne_group/Shared/AFM_Data/dna_damage/Cs137_irradiations")
dir_this_analysis = dir_base / "20260204-analysis-getting-back-into-the-project"
threshold_contour_length = 300

# Loading data

In [ ]:
assert dir_base.exists()
assert dir_this_analysis.exists()
dir_processed_data = dir_this_analysis / "output"
assert dir_processed_data.exists()
dir_results = dir_this_analysis / "analysis_results"
dir_results.mkdir(exist_ok=True)
assert dir_results.exists()

topo_files = list(dir_processed_data.glob("*/**/processed/*.topostats"))
print(f"found {len(topo_files)} topo files")

# Load the corresponding csv file
csv_grain_stats: Path = dir_processed_data / "grain_statistics.csv"
assert csv_grain_stats.exists()
df_grain_stats = pd.read_csv(csv_grain_stats)
print(f"grain stats columns: {df_grain_stats.columns}")

# convert some columns to nanometres
df_grain_stats["total_contour_length"] /= 1e-9

# convert basename to only use the parent directory
df_grain_stats["basename"] = df_grain_stats["basename"].apply(lambda x: Path(x).parent)

In [ ]:
print(df_grain_stats["basename"].value_counts())

# Sample vetting

In [ ]:
# plot contour length distributions
sns.stripplot(data=df_grain_stats, x="basename", y="total_contour_length", s=2)
sns.violinplot(data=df_grain_stats, x="basename", y="total_contour_length", inner=None)
plt.xticks(rotation=90)
plt.title("Contour length distributions")
plt.show()

# drop any rows with contour length less than a threshold
n_rows_before = len(df_grain_stats)
df_grain_stats = df_grain_stats[df_grain_stats["total_contour_length"] >= threshold_contour_length]
n_rows_after = len(df_grain_stats)
print(
    f"dropped {n_rows_before - n_rows_after} rows with contour length < {threshold_contour_length} nm."
    f"remaining rows: {n_rows_after}"
)

sns.stripplot(data=df_grain_stats, x="basename", y="total_contour_length", s=2)
sns.violinplot(data=df_grain_stats, x="basename", y="total_contour_length", inner=None)
plt.xticks(rotation=90)
plt.title("Contour length distributions")
plt.show()

In [ ]:
unanalysed_grain_collection: UnanalysedGrainCollection = construct_grains_collection_from_topostats_files(
    dir_to_save_cache_files=dir_results,
    dir_processed_data=dir_processed_data,
    df_grain_stats=df_grain_stats,
    bbox_padding=1,
    force_reload=False,
)
print(unanalysed_grain_collection)

In [ ]:
test = str(Path("file.txt").stem)
print(test)